# DOA Pipeline Demo

Streams a WAV file through **four parallel DOA chains** for comparison.

| Chain | Ring | Filter | freq_range | Notes |
|-------|------|--------|-----------|-------|
| A | Outer CH1–4 | none | [50, 357] Hz | below spatial aliasing limit |
| B | Inner CH5–8 | HP 400 Hz | [400, 808] Hz | wind removed; below aliasing limit |
| C | Outer CH1–4 | none | default [200, 4000] Hz | no freq restriction |
| D | Inner CH5–8 | none | default [200, 4000] Hz | no freq restriction, no HP |

**File:** 8 channels, 44100 Hz  
Outer ring aliasing limit ≈ 357 Hz · Inner ring aliasing limit ≈ 808 Hz

In [1]:
import sys
sys.path.insert(0, '..')  # make logic/ importable from notebooks/

import numpy as np
from logic.wav_loader import WavLoader
from logic.stft_processor import StftProcessor, StftChunk
from logic.doa_processor import DoaProcessor
from logic.filters import HighPassFilter


In [ ]:
# --- Config ---
WAV_FILE    = '../data/static_10m_000.wav'
SAMPLE_RATE = 44100
CHUNK_SIZE  = 8192
NPERSEG     = 512
NOVERLAP    = 256

OUTER_IDX = slice(0, 4)   # CH1-4
INNER_IDX = slice(4, 8)   # CH5-8

# Outer ring (CH1-4): 34 cm square
# Spatial aliasing limit: c/(2*d_max) = 343/(2*0.481) ≈ 357 Hz
L_outer = np.array([
    [-0.17,  0.17, -0.17,  0.17],
    [-0.17, -0.17,  0.17,  0.17],
    [ 0.00,  0.00,  0.00,  0.00],
])

# Inner ring (CH5-8): 15 cm square
# Spatial aliasing limit: c/(2*d_max) = 343/(2*0.212) ≈ 808 Hz
L_inner = np.array([
    [-0.075,  0.075, -0.075,  0.075],
    [-0.075, -0.075,  0.075,  0.075],
    [ 0.000,  0.000,  0.000,  0.000],
])

HP_CUTOFF_HZ = 400   # wind removal cutoff


In [3]:
# --- Processors (one WavLoader + StftProcessor per chain) ---
# Chain A: outer, raw, freq_range=[50,357]
wav_A  = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)
stft_A = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
doa_A  = DoaProcessor(mic_locs=L_outer, sampling_rate=SAMPLE_RATE, nfft=NPERSEG,
                       freq_range=[50, 357])

# Chain B: inner, HP@400Hz, freq_range=[400,808]
wav_B  = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)
stft_B = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
hp_B   = HighPassFilter(cutoff_hz=HP_CUTOFF_HZ, sampling_rate=SAMPLE_RATE, order=4)
doa_B  = DoaProcessor(mic_locs=L_inner, sampling_rate=SAMPLE_RATE, nfft=NPERSEG,
                       freq_range=[400, 808])

# Chain C: outer, raw, default freq_range
wav_C  = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)
stft_C = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
doa_C  = DoaProcessor(mic_locs=L_outer, sampling_rate=SAMPLE_RATE, nfft=NPERSEG)

# Chain D: inner, raw (no HP), default freq_range
wav_D  = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)
stft_D = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
doa_D  = DoaProcessor(mic_locs=L_inner, sampling_rate=SAMPLE_RATE, nfft=NPERSEG)


In [4]:
def _slice_stft(stft_proc, audio_gen, ch_slice):
    for chunk in stft_proc.process(audio_gen):
        yield StftChunk(freqs=chunk.freqs, times=chunk.times,
                        magnitudes=chunk.magnitudes[ch_slice],
                        sampling_rate=chunk.sampling_rate, timestamp=chunk.timestamp)

def _run_doa(stft_proc, audio_gen, doa_proc, ch_slice):
    return {d.timestamp: d.azimuth_deg
            for d in doa_proc.process(_slice_stft(stft_proc, audio_gen, ch_slice))}

# Pre-compute chains B, C, D (keyed by timestamp)
doa_B_map = _run_doa(stft_B, hp_B.process(wav_B.stream()), doa_B, INNER_IDX)
doa_C_map = _run_doa(stft_C, wav_C.stream(), doa_C, OUTER_IDX)
doa_D_map = _run_doa(stft_D, wav_D.stream(), doa_D, INNER_IDX)

SEP = '─' * 90
print(SEP)
print(f"{'time':>8}  {'A outer<357':>12}  {'B inner HP':>12}  {'C outer raw':>12}  {'D inner raw':>12}")
print(f"{'':>8}  {'[50-357Hz]':>12}  {'[400-808Hz]':>12}  {'[200-4kHz]':>12}  {'[200-4kHz]':>12}")
print(SEP)

def stft_A_stream():
    for chunk in stft_A.process(wav_A.stream()):
        print(f"{chunk.timestamp:7.3f}s  AudioChunk → StftChunk  shape={chunk.magnitudes.shape}"
              f"  freqs={chunk.freqs[0]:.0f}–{chunk.freqs[-1]:.0f} Hz")
        yield StftChunk(freqs=chunk.freqs, times=chunk.times,
                        magnitudes=chunk.magnitudes[OUTER_IDX],
                        sampling_rate=chunk.sampling_rate, timestamp=chunk.timestamp)

for d_A in doa_A.process(stft_A_stream()):
    t = d_A.timestamp
    a = d_A.azimuth_deg
    b = doa_B_map.get(t, float('nan'))
    c = doa_C_map.get(t, float('nan'))
    d = doa_D_map.get(t, float('nan'))
    print(f"{t:7.3f}s  {a:>11.1f}°  {b:>11.1f}°  {c:>11.1f}°  {d:>11.1f}°")

print(SEP)


────────────────────────────────────────────────────────────────────────────
timestamp      outer DOA     inner DOA  note
────────────────────────────────────────────────────────────────────────────
  0.000s  AudioChunk → StftChunk  shape=(8, 257, 33)  freqs=0–22050 Hz
  0.000s   outer=  82.0°   inner= 358.0°  ← inner HP+filtered, freq-limited
  0.186s  AudioChunk → StftChunk  shape=(8, 257, 33)  freqs=0–22050 Hz
  0.186s   outer= 126.0°   inner= 358.0°  ← inner HP+filtered, freq-limited
  0.372s  AudioChunk → StftChunk  shape=(8, 257, 33)  freqs=0–22050 Hz
  0.372s   outer= 131.0°   inner= 344.0°  ← inner HP+filtered, freq-limited
  0.557s  AudioChunk → StftChunk  shape=(8, 257, 33)  freqs=0–22050 Hz
  0.557s   outer= 329.0°   inner= 359.0°  ← inner HP+filtered, freq-limited
  0.743s  AudioChunk → StftChunk  shape=(8, 257, 33)  freqs=0–22050 Hz
  0.743s   outer= 288.0°   inner=  15.0°  ← inner HP+filtered, freq-limited
  0.929s  AudioChunk → StftChunk  shape=(8, 257, 27)  freqs=0–2205